# Step 10 — SYSU Phase-2 Segmented Paper-like Run

This notebook does **not** edit source code. Apply the Step 10 patch first, then use this notebook to run Phase-2 in resumable segments.

Typical first run:

- Load existing SYSU Phase-1 checkpoint `model_20.pth`.
- Run Phase-2 from epoch 0 to epoch 29 by setting `SEGMENT_END_EPOCH = 30`.
- Save `phase2_state_5.pth`, `phase2_state_10.pth`, ..., `phase2_state_30.pth`.

Typical next run:

- Set `BASELINE_PHASE2_STATE_PATH` and `UPR_PHASE2_STATE_PATH` to the saved `phase2_state_30.pth` files.
- Set `SEGMENT_END_EPOCH = 60`.
- Continue Phase-2 from epoch 30 to epoch 59.


## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [1]:
from pathlib import Path
from datetime import datetime

CFG = {
    "GITHUB_OWNER": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",

    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "SYSU_SOURCE": "/kaggle/input/datasets/coconutjean/sysumm01",  # empty = auto-detect in /kaggle/input
    "PHASE1_CKPT_PATH": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth",
    "SYSU_PREPROCESSED_SOURCE":"",

    # First paper-like segment. Change to 60/90/120 for later segments.
    "SEGMENT_END_EPOCH": 30,
    "SAVE_PHASE2_EVERY": 5,
    "RELATION_STATS_EVERY": 5,

    "LR": 0.0003,
    "MILESTONES": "30 70",
    "BATCH_PIDNUM": 8,
    "PID_NUMSAMPLE": 4,
    "TEST_BATCH": 128,
    "NUM_WORKERS": 4,
    "SEED": 1,

    "UPR_BETA": 0.1,
    "UPR_GAMMA": 0.0,
    "UPR_WARMUP_EPOCH": 2,
    "UPR_FILTER_START_EPOCH": 2,
    "UPR_FILTER_END_EPOCH": 10,
    "UPR_FILTER_START_RATIO": 0.55,
    "UPR_FILTER_END_RATIO": 1.0,
    "UPR_FILTER_MIN_PAIRS": 40,

    # Later segments only. First run: keep empty.
    "BASELINE_PHASE2_STATE_PATH": "",
    "UPR_PHASE2_STATE_PATH": "",

    "PUSH_SMALL_BACKUP_TO_GIT": True,
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "GIT_USER_NAME": "Kaggle Bot",
    "GIT_USER_EMAIL": "kaggle-bot@example.com",
}

RUN_SUFFIX = datetime.utcnow().strftime("sysu_seg_%Y%m%d_%H%M%S")
print("RUN_SUFFIX =", RUN_SUFFIX)
print("phase1 checkpoint exists:", Path(CFG["PHASE1_CKPT_PATH"]).exists())
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
print("repo_dir =", repo_dir)


RUN_SUFFIX = sysu_seg_20260702_012029
phase1 checkpoint exists: True
repo_dir = /kaggle/working/UIT.CS2309


/tmp/ipykernel_58/381123704.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_SUFFIX = datetime.utcnow().strftime("sysu_seg_%Y%m%d_%H%M%S")


## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [2]:
import subprocess
from pathlib import Path

repo_url = f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True)

subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, check=True)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)


Cloning into '/kaggle/working/UIT.CS2309'...


d25ecbd


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## Block 2 — Install requirements, apply compatibility patches, prepare SYSU-MM01

In [3]:
%%bash
set -e
cd /kaggle/working/UIT.CS2309/WSL_ReID
pip install -q -r requirements-kaggle.txt
python scripts/apply_kaggle_compat_patches.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
No compatibility patches were needed.


In [4]:
# import subprocess
# from pathlib import Path

# wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
# cmd = ["python", "scripts/prepare_sysu_kaggle.py", "--data-root", CFG["DATA_ROOT"]]
# if CFG["SYSU_SOURCE"]:
#     cmd += ["--sysu-source", CFG["SYSU_SOURCE"]]
# print("+", " ".join(cmd))
# subprocess.run(cmd, cwd=wsl_dir, check=True)
# subprocess.run(["python", "scripts/check_sysu_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=wsl_dir, check=True)


+ python scripts/prepare_sysu_kaggle.py --data-root /kaggle/working/VIREID_Dataset --sysu-source /kaggle/input/datasets/coconutjean/sysumm01
SYSU source: /kaggle/input/datasets/coconutjean/sysumm01
SYSU destination: /kaggle/working/VIREID_Dataset/SYSU-MM01
SYSU train+val identities: 395
SYSU RGB train images: 22258
SYSU IR train images: 11909


Processing SYSU ir images: 100%|██████████| 11909/11909 [01:25<00:00, 139.98it/s]


Created SYSU preprocessed numpy files.
SYSU prepared at: /kaggle/working/VIREID_Dataset/SYSU-MM01
train_rgb_info: (22258, 3) train_rgb_images: (22258, 288, 144, 3)
train_ir_info: (11909, 3) train_ir_images: (11909, 288, 144, 3)
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== SYSU-MM01 layout ===
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam1 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam2 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam3 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam4 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam5 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam6 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/exp OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/train_rgb_modified_img.npy OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/train_rgb_info.npy OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/train_ir_modified_img.npy OK
/kaggle/working/VIREID_Dat

CompletedProcess(args=['python', 'scripts/check_sysu_env.py', '--data-root', '/kaggle/working/VIREID_Dataset'], returncode=0)

In [ ]:
from pathlib import Path
import shutil
import subprocess

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
data_root = Path(CFG["DATA_ROOT"])
dst = data_root / "SYSU-MM01"

# Set these if you know exact paths. Leave empty to auto-detect.
SYSU_RAW_SOURCE = CFG.get("SYSU_SOURCE", "")  # raw SYSU-MM01 with cam1..cam6/exp
SYSU_PREPROCESSED_SOURCE = CFG.get(
    "SYSU_PREPROCESSED_SOURCE",
    "/kaggle/input/uit-cs2309-sysu-preprocessed/SYSU-MM01"
)

required_cameras = ["cam1", "cam2", "cam3", "cam4", "cam5", "cam6"]
required_preprocessed = [
    "train_rgb_modified_img.npy",
    "train_rgb_info.npy",
    "train_ir_modified_img.npy",
    "train_ir_info.npy",
]
optional_preprocessed = ["rgbi2p.pk", "iri2p.pk"]

def find_raw_sysu(input_root="/kaggle/input"):
    candidates = []
    for p in Path(input_root).rglob("SYSU-MM01"):
        if p.is_dir() and (p / "exp").exists() and all((p / cam).exists() for cam in required_cameras):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Could not find raw SYSU-MM01 under /kaggle/input. Set SYSU_RAW_SOURCE.")
    return sorted(candidates, key=lambda x: len(str(x)))[0]

def find_preprocessed_sysu(input_root="/kaggle/input"):
    candidates = []
    for p in Path(input_root).rglob("SYSU-MM01"):
        if p.is_dir() and all((p / name).exists() for name in required_preprocessed):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(
            "Could not find preprocessed SYSU-MM01 .npy files under /kaggle/input. "
            "Set SYSU_PREPROCESSED_SOURCE or upload the cached .npy dataset."
        )
    return sorted(candidates, key=lambda x: len(str(x)))[0]

def link_or_copy(src: Path, dst_path: Path, copy=False):
    if dst_path.exists() or dst_path.is_symlink():
        if dst_path.is_symlink() or dst_path.is_file():
            dst_path.unlink()
        else:
            shutil.rmtree(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if copy:
        if src.is_dir():
            shutil.copytree(src, dst_path)
        else:
            shutil.copy2(src, dst_path)
    else:
        dst_path.symlink_to(src, target_is_directory=src.is_dir())

raw_src = Path(SYSU_RAW_SOURCE) if SYSU_RAW_SOURCE else find_raw_sysu()
pre_src = Path(SYSU_PREPROCESSED_SOURCE) if SYSU_PREPROCESSED_SOURCE else find_preprocessed_sysu()

if not raw_src.exists():
    raise FileNotFoundError(f"Raw SYSU source not found: {raw_src}")
if not pre_src.exists():
    raise FileNotFoundError(f"Preprocessed SYSU source not found: {pre_src}")

print("Raw SYSU source:", raw_src)
print("Preprocessed SYSU source:", pre_src)
print("Destination:", dst)

dst.mkdir(parents=True, exist_ok=True)

# Link raw cameras and exp.
for cam in required_cameras:
    link_or_copy(raw_src / cam, dst / cam, copy=False)

link_or_copy(raw_src / "exp", dst / "exp", copy=False)

# Link cached preprocessed files.
for name in required_preprocessed + optional_preprocessed:
    src_file = pre_src / name
    if src_file.exists():
        link_or_copy(src_file, dst / name, copy=False)

# Verify environment.
subprocess.run(
    ["python", "scripts/check_sysu_env.py", "--data-root", str(data_root)],
    cwd=wsl_dir,
    check=True,
)

## Block 3 — Verify Step 10 code markers

In [5]:
from pathlib import Path
import subprocess

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
required = [
    "main.py",
    "models/__init__.py",
    "scripts/run_sysu_phase2_segment_t4x2.sh",
    "scripts/collect_relation_stats.py",
    "scripts/prepare_sysu_kaggle.py",
    "scripts/check_sysu_env.py",
]
missing = [p for p in required if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))

checks = {
    "main.py": ["--phase2-state-path", "--save-phase2-every", "--phase2-state-dir", "load_phase2_state", "save_phase2_state"],
    "models/__init__.py": ["def save_phase2_state", "def load_phase2_state", "phase2_next_epoch", "optimizer_phase2"],
    "scripts/run_sysu_phase2_segment_t4x2.sh": ["SEGMENT_END_EPOCH", "PHASE1_CKPT_PATH", "BASELINE_PHASE2_STATE_PATH", "UPR_PHASE2_STATE_PATH", "--save-phase2-every"],
}
for rel, markers in checks.items():
    text = (wsl_dir / rel).read_text()
    for marker in markers:
        if marker not in text:
            raise RuntimeError(f"Missing marker {marker!r} in {rel}")
print("Step 10 marker check OK.")

subprocess.run(["python", "-m", "py_compile", "main.py", "models/__init__.py", "scripts/collect_relation_stats.py"], cwd=wsl_dir, check=True)
subprocess.run(["bash", "-n", "scripts/run_sysu_phase2_segment_t4x2.sh"], cwd=wsl_dir, check=True)
print("Static checks OK.")


Step 10 marker check OK.
Static checks OK.


## Block 4 — Start first / next segment on two GPUs

In [6]:
%%time
import os, subprocess
from pathlib import Path

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"

if CFG["BASELINE_PHASE2_STATE_PATH"] or CFG["UPR_PHASE2_STATE_PATH"]:
    assert Path(CFG["BASELINE_PHASE2_STATE_PATH"]).exists(), CFG["BASELINE_PHASE2_STATE_PATH"]
    assert Path(CFG["UPR_PHASE2_STATE_PATH"]).exists(), CFG["UPR_PHASE2_STATE_PATH"]
else:
    assert Path(CFG["PHASE1_CKPT_PATH"]).exists(), CFG["PHASE1_CKPT_PATH"]

env = os.environ.copy()
env.update({
    "DATA_ROOT": CFG["DATA_ROOT"],
    "PHASE1_CKPT_PATH": CFG["PHASE1_CKPT_PATH"],
    "BASELINE_PHASE2_STATE_PATH": CFG["BASELINE_PHASE2_STATE_PATH"],
    "UPR_PHASE2_STATE_PATH": CFG["UPR_PHASE2_STATE_PATH"],
    "SEGMENT_END_EPOCH": str(CFG["SEGMENT_END_EPOCH"]),
    "SAVE_PHASE2_EVERY": str(CFG["SAVE_PHASE2_EVERY"]),
    "RELATION_STATS_EVERY": str(CFG["RELATION_STATS_EVERY"]),
    "RUN_SUFFIX": RUN_SUFFIX,
    "LR": str(CFG["LR"]),
    "BATCH_PIDNUM": str(CFG["BATCH_PIDNUM"]),
    "PID_NUMSAMPLE": str(CFG["PID_NUMSAMPLE"]),
    "TEST_BATCH": str(CFG["TEST_BATCH"]),
    "NUM_WORKERS": str(CFG["NUM_WORKERS"]),
    "MILESTONES": CFG["MILESTONES"],
    "SEED": str(CFG["SEED"]),
    "UPR_BETA": str(CFG["UPR_BETA"]),
    "UPR_GAMMA": str(CFG["UPR_GAMMA"]),
    "UPR_WARMUP_EPOCH": str(CFG["UPR_WARMUP_EPOCH"]),
    "UPR_FILTER_START_EPOCH": str(CFG["UPR_FILTER_START_EPOCH"]),
    "UPR_FILTER_END_EPOCH": str(CFG["UPR_FILTER_END_EPOCH"]),
    "UPR_FILTER_START_RATIO": str(CFG["UPR_FILTER_START_RATIO"]),
    "UPR_FILTER_END_RATIO": str(CFG["UPR_FILTER_END_RATIO"]),
    "UPR_FILTER_MIN_PAIRS": str(CFG["UPR_FILTER_MIN_PAIRS"]),
})

subprocess.run(["bash", "scripts/run_sysu_phase2_segment_t4x2.sh"], cwd=wsl_dir, env=env, check=True)
print(Path("/kaggle/working/sysu_segment_runtime_info.json").read_text())


[SYSU segment] repo: /kaggle/working/UIT.CS2309/WSL_ReID
[SYSU segment] RUN_SUFFIX: sysu_seg_20260702_012029
[SYSU segment] SEGMENT_END_EPOCH: 30
[SYSU segment] DATA_ROOT: /kaggle/working/VIREID_Dataset
[SYSU segment] PHASE1_CKPT_PATH: /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth
[SYSU segment] BASELINE_PHASE2_STATE_PATH: <empty>
[SYSU segment] UPR_PHASE2_STATE_PATH: <empty>
Started SYSU segment jobs:
baseline PID: 140
UPR PID:      141
runtime info: /kaggle/working/sysu_segment_runtime_info.json
{
  "run_suffix": "sysu_seg_20260702_012029",
  "segment_end_epoch": 30,
  "save_phase2_every": 5,
  "relation_stats_every": 5,
  "data_root": "/kaggle/working/VIREID_Dataset",
  "phase1_ckpt_path": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth",
  "baseline_phase2_state_path": "",
  "upr_phase2_state_path": "",
  "baseline_save_path": "baseline_sysu_p2s30_sysu_seg_20260702_012029",
  "upr_save_path": "upr_v02_sysu_p2s30_sysu_seg_202607

## Block 5 — Monitor logs and GPUs


In [7]:
%%time
%%bash
set +e
python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print('baseline_log=', info['baseline_log'])
print('upr_log=', info['upr_log'])
PY
nvidia-smi
for LOG in $(python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print(info['baseline_log'])
print(info['upr_log'])
PY
); do
  echo "=============================="
  echo "$LOG"
  tail -n 80 "$LOG" || true
done


baseline_log= /kaggle/working/run_logs/baseline_sysu_p2s30_sysu_seg_20260702_012029.log
upr_log= /kaggle/working/run_logs/upr_v02_sysu_p2s30_sysu_seg_20260702_012029.log
Thu Jul  2 01:25:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  1536

## Block 6 — Wait until both jobs finish


In [8]:
%%time
%%bash
set +e

HEARTBEAT_LOG="/kaggle/working/sysu_segment_wait_heartbeat.log"
exec > >(tee -a "$HEARTBEAT_LOG") 2>&1

PIDS="/kaggle/working/pids/sysu_baseline_segment.pid /kaggle/working/pids/sysu_upr_segment.pid"
RUNTIME_JSON="/kaggle/working/sysu_segment_runtime_info.json"

if [ ! -f "$RUNTIME_JSON" ]; then
  echo "ERROR: missing $RUNTIME_JSON. Run Block 4 first."
  exit 1
fi

BASE_LOG=$(python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print(info.get('baseline_log',''))
PY
)
UPR_LOG=$(python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print(info.get('upr_log',''))
PY
)
BASE_SAVE=$(python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print(info.get('baseline_save_path',''))
PY
)
UPR_SAVE=$(python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print(info.get('upr_save_path',''))
PY
)
SEGMENT_END=$(python - <<'PY'
import json
from pathlib import Path
info=json.loads(Path('/kaggle/working/sysu_segment_runtime_info.json').read_text())
print(info.get('segment_end_epoch',''))
PY
)

echo "============================================================"
echo "SYSU segmented Phase-2 wait with heartbeat"
echo "Started at UTC: $(date -u '+%Y-%m-%d %H:%M:%S')"
echo "segment_end_epoch: $SEGMENT_END"
echo "baseline_save_path: $BASE_SAVE"
echo "upr_save_path:      $UPR_SAVE"
echo "baseline_log:       $BASE_LOG"
echo "upr_log:            $UPR_LOG"
echo "heartbeat log:      $HEARTBEAT_LOG"
echo "Tracking PIDs:"
for PID_FILE in $PIDS; do
  if [ -f "$PID_FILE" ]; then
    echo "  $PID_FILE -> $(cat "$PID_FILE")"
  else
    echo "  missing: $PID_FILE"
  fi
done
echo "============================================================"

ITER=0
SLEEP_SECONDS=60

while true; do
  ITER=$((ITER + 1))
  RUNNING=0

  echo ""
  echo "================ HEARTBEAT $ITER | UTC $(date -u '+%Y-%m-%d %H:%M:%S') ================"

  echo "--- PID status ---"
  for PID_FILE in $PIDS; do
    if [ ! -f "$PID_FILE" ]; then
      echo "missing pid file: $PID_FILE"
      continue
    fi
    PID=$(cat "$PID_FILE")
    if kill -0 "$PID" 2>/dev/null; then
      RUNNING=1
      ps -p "$PID" -o pid,ppid,etime,stat,cmd || true
    else
      echo "finished or not found: PID=$PID ($PID_FILE)"
    fi
  done

  echo "--- GPU summary ---"
  nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw --format=csv,noheader || true

  echo "--- GPU processes ---"
  nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader || true

  echo "--- log file status ---"
  for LOG in "$BASE_LOG" "$UPR_LOG"; do
    if [ -f "$LOG" ]; then
      echo "$(basename "$LOG") | size=$(du -h "$LOG" | awk '{print $1}') | modified=$(date -u -r "$LOG" '+%Y-%m-%d %H:%M:%S')"
    else
      echo "missing log: $LOG"
    fi
  done

  echo "--- recent baseline progress ---"
  if [ -f "$BASE_LOG" ]; then
    grep -E "start phase2|Epoch:|R1:|Best_R1|mAP|mINP|phase2_state|save_phase2|Traceback|ERROR|Error|error|Killed|out of memory|CUDA" "$BASE_LOG" | tail -n 25 || tail -n 25 "$BASE_LOG" || true
  fi

  echo "--- recent UPR progress ---"
  if [ -f "$UPR_LOG" ]; then
    grep -E "start phase2|Epoch:|R1:|Best_R1|mAP|mINP|phase2_state|save_phase2|Traceback|ERROR|Error|error|Killed|out of memory|CUDA" "$UPR_LOG" | tail -n 25 || tail -n 25 "$UPR_LOG" || true
  fi

  echo "--- latest phase2 state checkpoints ---"
  find /kaggle/working/UIT.CS2309/saved_sysu_resnet -path "*${BASE_SAVE}*/phase2_states/*.pth" -printf "%TY-%Tm-%Td %TH:%TM %p\n" 2>/dev/null | sort | tail -n 5 || true
  find /kaggle/working/UIT.CS2309/saved_sysu_resnet -path "*${UPR_SAVE}*/phase2_states/*.pth" -printf "%TY-%Tm-%Td %TH:%TM %p\n" 2>/dev/null | sort | tail -n 5 || true

  if [ "$RUNNING" -eq 0 ]; then
    echo "All tracked jobs finished at UTC $(date -u '+%Y-%m-%d %H:%M:%S')."
    break
  fi

  echo "Sleeping ${SLEEP_SECONDS}s before next heartbeat..."
  sleep "$SLEEP_SECONDS"
done

echo ""
echo "================ FINAL LOG TAILS ================"
echo "--- baseline final tail ---"
[ -f "$BASE_LOG" ] && tail -n 80 "$BASE_LOG" || true
echo "--- UPR final tail ---"
[ -f "$UPR_LOG" ] && tail -n 80 "$UPR_LOG" || true

Process is interrupted.
CPU times: user 920 ms, sys: 325 ms, total: 1.25 s
Wall time: 7h 11min 3s


## Block 7 — Collect metrics and checkpoint paths


In [9]:
import json, re, csv, subprocess
from pathlib import Path

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
info = json.loads(Path("/kaggle/working/sysu_segment_runtime_info.json").read_text())
saved_root = wsl_dir.parent / "saved_sysu_resnet"
rows = []

def parse_best(log_text):
    matches = re.findall(r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)", log_text)
    if not matches:
        return "", "", "", "", ""
    return max(matches, key=lambda x: float(x[3]))

for tag, save_path in [("baseline", info["baseline_save_path"]), ("upr_v02", info["upr_save_path"])]:
    run_dir = saved_root / save_path
    log_path = run_dir / "log" / "log.txt"
    stats_dir = run_dir / "relation_stats"
    stats_csv = stats_dir / "relation_stats_summary.csv"
    if stats_dir.exists():
        subprocess.run(["python", "scripts/collect_relation_stats.py", "--stats-dir", str(stats_dir), "--csv-output", str(stats_csv)], cwd=wsl_dir, check=True)
    log_text = log_path.read_text() if log_path.exists() else ""
    r1,r10,r20,map_,minp = parse_best(log_text)
    last_stats = {}
    if stats_csv.exists():
        with stats_csv.open() as f:
            rr=list(csv.DictReader(f))
            if rr: last_stats=rr[-1]
    state_dir = run_dir / "phase2_states"
    state_files = sorted(state_dir.glob("phase2_state_*.pth")) if state_dir.exists() else []
    latest_state = str(state_files[-1]) if state_files else ""
    rows.append({
        "tag": tag,
        "save_path": save_path,
        "segment_end_epoch": info["segment_end_epoch"],
        "best_r1_by_map": r1,
        "best_r10_by_map": r10,
        "best_r20_by_map": r20,
        "best_map": map_,
        "best_minp": minp,
        "last_common_accuracy": last_stats.get("common_accuracy", ""),
        "last_mutual_match_ratio": last_stats.get("mutual_match_ratio", ""),
        "last_num_common": last_stats.get("num_common", ""),
        "latest_phase2_state": latest_state,
    })

out = Path("/kaggle/working/sysu_segment_summary.csv")
with out.open("w", newline="") as f:
    writer=csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader(); writer.writerows(rows)
print(out.read_text())


/kaggle/working/UIT.CS2309/saved_sysu_resnet/baseline_sysu_p2s30_sysu_seg_20260702_012029/relation_stats/relation_stats_summary.csv
/kaggle/working/UIT.CS2309/saved_sysu_resnet/upr_v02_sysu_p2s30_sysu_seg_20260702_012029/relation_stats/relation_stats_summary.csv
tag,save_path,segment_end_epoch,best_r1_by_map,best_r10_by_map,best_r20_by_map,best_map,best_minp,last_common_accuracy,last_mutual_match_ratio,last_num_common,latest_phase2_state
baseline,baseline_sysu_p2s30_sysu_seg_20260702_012029,30,0.5410,0.9034,0.9621,0.5209,0.3778,0.9680851063829787,0.7480106100795756,282,/kaggle/working/UIT.CS2309/saved_sysu_resnet/baseline_sysu_p2s30_sysu_seg_20260702_012029/phase2_states/phase2_state_5.pth
upr_v02,upr_v02_sysu_p2s30_sysu_seg_20260702_012029,30,0.5046,0.8890,0.9498,0.5053,0.3797,0.974025974025974,0.8213333333333334,308,/kaggle/working/UIT.CS2309/saved_sysu_resnet/upr_v02_sysu_p2s30_sysu_seg_20260702_012029/phase2_states/phase2_state_5.pth



## Block 8 — Archive logs, summaries, and Phase-2 state checkpoints


In [10]:
from pathlib import Path
import json, shutil, subprocess

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
info = json.loads(Path("/kaggle/working/sysu_segment_runtime_info.json").read_text())
saved_root = wsl_dir.parent / "saved_sysu_resnet"
artifact_dir = Path(f"/kaggle/working/artifacts_sysu_segment_{info['run_suffix']}")
if artifact_dir.exists(): shutil.rmtree(artifact_dir)
(artifact_dir/"logs").mkdir(parents=True)
(artifact_dir/"relation_stats").mkdir(parents=True)
(artifact_dir/"phase2_states").mkdir(parents=True)

for p in [Path('/kaggle/working/sysu_segment_runtime_info.json'), Path('/kaggle/working/sysu_segment_summary.csv')]:
    if p.exists(): shutil.copy2(p, artifact_dir/p.name)

for tag, save_path in [("baseline", info["baseline_save_path"]), ("upr_v02", info["upr_save_path"])]:
    run_dir = saved_root / save_path
    log_file = run_dir / "log" / "log.txt"
    if log_file.exists(): shutil.copy2(log_file, artifact_dir/"logs"/f"{save_path}_log.txt")
    stats_csv = run_dir / "relation_stats" / "relation_stats_summary.csv"
    if stats_csv.exists(): shutil.copy2(stats_csv, artifact_dir/"relation_stats"/f"{save_path}_relation_stats_summary.csv")
    state_dir = run_dir / "phase2_states"
    if state_dir.exists():
        dst = artifact_dir/"phase2_states"/save_path
        shutil.copytree(state_dir, dst, dirs_exist_ok=True)

tar_path = Path(f"/kaggle/working/artifacts_sysu_segment_{info['run_suffix']}.tar.gz")
if tar_path.exists(): tar_path.unlink()
subprocess.run(["tar", "-czf", str(tar_path), "-C", "/kaggle/working", artifact_dir.name], check=True)
print("Artifact:", tar_path)
print("Size MB:", tar_path.stat().st_size/1024/1024)


Artifact: /kaggle/working/artifacts_sysu_segment_sysu_seg_20260702_012029.tar.gz
Size MB: 2569.5857858657837


## Block 9 — Optional: push small backup to GitHub using old git push method


In [11]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import os, json, shutil, subprocess, textwrap, hashlib

if not CFG.get("PUSH_SMALL_BACKUP_TO_GIT", False):
    print("Git backup disabled.")
else:
    repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
    info = json.loads(Path("/kaggle/working/sysu_segment_runtime_info.json").read_text())
    run_id = info["run_suffix"]
    src_artifact_dir = Path(f"/kaggle/working/artifacts_sysu_segment_{run_id}")
    tar_path = Path(f"/kaggle/working/artifacts_sysu_segment_{run_id}.tar.gz")
    backup_dir = repo_dir / "experiments" / "kaggle_runs" / run_id
    if backup_dir.exists(): shutil.rmtree(backup_dir)
    backup_dir.mkdir(parents=True, exist_ok=True)

    for sub in ["logs", "relation_stats"]:
        if (src_artifact_dir/sub).exists(): shutil.copytree(src_artifact_dir/sub, backup_dir/sub, dirs_exist_ok=True)
    for p in src_artifact_dir.glob("*.csv"):
        shutil.copy2(p, backup_dir/p.name)
    for p in src_artifact_dir.glob("*.json"):
        shutil.copy2(p, backup_dir/p.name)

    def sha256_file(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for chunk in iter(lambda: f.read(1024*1024), b''):
                h.update(chunk)
        return h.hexdigest()

    large_files=[]
    if tar_path.exists(): large_files.append(tar_path)
    phase_dir = src_artifact_dir/"phase2_states"
    if phase_dir.exists():
        large_files.extend(phase_dir.rglob("*.pth"))
    manifest={
        "run_id": run_id,
        "branch": CFG["BRANCH"],
        "artifact_tar": str(tar_path) if tar_path.exists() else None,
        "large_files": [
            {"source_path": str(p), "name": p.name, "size_mb": round(p.stat().st_size/1024/1024,2), "sha256": sha256_file(p)}
            for p in large_files
        ],
        "note": "Large .pth/.tar.gz files are not committed by default. Upload artifact tar to Kaggle Dataset/Input for later segments."
    }
    (backup_dir/"manifest.json").write_text(json.dumps(manifest, indent=2))
    (backup_dir/"README.md").write_text(f"# SYSU segment backup: {run_id}\n\nSmall logs/CSV are committed. Large phase2 checkpoints are listed in manifest.json.\n")

    token = UserSecretsClient().get_secret(CFG["GITHUB_TOKEN_SECRET"]).strip()
    askpass = Path(CFG["WORK_DIR"]) / "git_askpass_sysu_segment.sh"
    script = "#!/bin/sh\ncase \"$1\" in\n  *Username*) echo \"{}\" ;;\n  *Password*) echo \"{}\" ;;\n  *) echo \"\" ;;\nesac\n".format(CFG['GITHUB_OWNER'], token)
    askpass.write_text(script)
    askpass.chmod(0o700)
    env=os.environ.copy(); env["GIT_ASKPASS"]=str(askpass); env["GIT_TERMINAL_PROMPT"]="0"
    try:
        subprocess.run(["git","config","user.name",CFG["GIT_USER_NAME"]], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git","config","user.email",CFG["GIT_USER_EMAIL"]], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git","fetch","origin",CFG["BRANCH"]], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git","checkout",CFG["BRANCH"]], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git","pull","--rebase","origin",CFG["BRANCH"]], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git","add","-f",str(backup_dir.relative_to(repo_dir))], cwd=repo_dir, env=env, check=True)
        diff = subprocess.run(["git","diff","--cached","--quiet"], cwd=repo_dir, env=env)
        if diff.returncode != 0:
            subprocess.run(["git","commit","-m",f"exp: add SYSU phase2 segment backup {run_id}"], cwd=repo_dir, env=env, check=True)
            subprocess.run(["git","push","origin",CFG["BRANCH"]], cwd=repo_dir, env=env, check=True)
        else:
            print("No staged changes.")
        print(f"Pushed: https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}/tree/{CFG['BRANCH']}/experiments/kaggle_runs/{run_id}")
    finally:
        try: askpass.unlink()
        except FileNotFoundError: pass


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
Already on 'feature/upr-cre'


Your branch is up to date with 'origin/feature/upr-cre'.


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD


Already up to date.
[feature/upr-cre 924b7d2] exp: add SYSU phase2 segment backup sysu_seg_20260702_012029
 8 files changed, 697 insertions(+)
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/README.md
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/logs/baseline_sysu_p2s30_sysu_seg_20260702_012029_log.txt
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/logs/upr_v02_sysu_p2s30_sysu_seg_20260702_012029_log.txt
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/manifest.json
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/relation_stats/baseline_sysu_p2s30_sysu_seg_20260702_012029_relation_stats_summary.csv
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/relation_stats/upr_v02_sysu_p2s30_sysu_seg_20260702_012029_relation_stats_summary.csv
 create mode 100644 experiments/kaggle_runs/sysu_seg_20260702_012029/sysu_segment_runtime_info.json
 create mode 100644 experiments

To https://github.com/TranTruongMMCII/UIT.CS2309.git
   d25ecbd..924b7d2  feature/upr-cre -> feature/upr-cre
